# Cosmos 3 Reasoner on HATRec — Evidence-grounded Assembly Report

This notebook sends **one anonymized HATRec industrial assembly video** to
`nvidia/Cosmos3-Nano` and produces a document similar to a long-form sports
video analysis, adapted for industrial assembly:

- temporal group analysis;
- visible hand actions and hand-object interactions;
- tools, parts and state changes;
- observed vs inferred vs unknown evidence;
- task classification, confidence and alternatives;
- execution/safety observations;
- full-video synthesis;
- black-video and reversed-video causal controls.

The default run is a one-video pilot. Ground-truth labels are hidden from the
model and revealed only after inference. The source dataset is research-only
under **CC BY-NC-ND 4.0**; do not redistribute its videos or derived video
controls.

**Recommended runtime:** Linux with an H100 80 GB or RTX PRO 6000-class GPU.
Cosmos 3 Nano is a 16B checkpoint. A T4 runtime is not the intended target.


## 1. Install the official Cosmos 3 Transformers path

In [ ]:
import subprocess, sys

packages = [
    'transformers>=5.11.0', 'accelerate>=1.10', 'av>=14.0',
    'safetensors>=0.8.0', 'requests>=2.31', 'tqdm>=4.66',
    'pandas>=2.0', 'scikit-learn>=1.4', 'matplotlib>=3.7',
    'seaborn>=0.13'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print('Dependencies installed. If Transformers was upgraded in a live runtime, restart once if import fails.')


## 2. Configuration — one video by default

In [ ]:
from pathlib import Path
import os, json, math, re, shutil, time, zipfile, hashlib, random

MODEL_ID = 'nvidia/Cosmos3-Nano'
DATASET_SLUG = 'ayoznur/hatrec-video-dataset'
DATASET_URL = f'https://www.kaggle.com/api/v1/datasets/download/{DATASET_SLUG}'

if Path('/kaggle/working').exists():
    WORK = Path('/kaggle/working/hatrec_cosmos3')
elif Path('/content').exists():
    WORK = Path('/content/hatrec_cosmos3')
else:
    WORK = Path.cwd() / 'hatrec_cosmos3_work'

DATA_DIR = WORK / 'data'
OUTPUT_DIR = WORK / 'outputs'
BLIND_DIR = WORK / 'blind_media'
CHUNK_DIR = WORK / 'chunks'
for directory in (DATA_DIR, OUTPUT_DIR, BLIND_DIR, CHUNK_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Pilot controls
PILOT_CYCLE = 0
PILOT_TASK = 0
VIDEO_FPS_FOR_MODEL = 2          # NVIDIA's official video example uses fps=2
SEGMENT_SECONDS = 8.0
MAX_SEGMENTS = 8
MAX_NEW_TOKENS_SEGMENT = 700
MAX_NEW_TOKENS_SYNTHESIS = 1100
RUN_CAUSAL_CONTROLS = True

# Optional full closed-set benchmark; leave False for the first pilot.
RUN_FULL_BENCHMARK = False
FULL_BENCHMARK_LIMIT = 70

TASK_CATALOG = {
    0: 'Assembling the spring',
    1: 'Placing the white plastic part',
    2: 'Screwing-1',
    3: 'Inflating the valve',
    4: 'Placing the black plastic part',
    5: 'Screwing-2',
    6: 'Fixing the cable',
}

print('work:', WORK)
print('model:', MODEL_ID)
print('pilot:', {'cycle': PILOT_CYCLE, 'task': PILOT_TASK})


## 3. Download HATRec and build a verified manifest

In [ ]:
import requests
from tqdm.auto import tqdm

attached = (list(Path('/kaggle/input').rglob('Cycle_0_task_0.mp4'))
            if Path('/kaggle/input').exists() else [])
if attached:
    search_root = Path('/kaggle/input')
    print('Using attached Kaggle dataset:', attached[0])
else:
    archive = WORK / 'hatrec-video-dataset.zip'
    if not archive.exists():
        with requests.get(DATASET_URL, stream=True, timeout=120) as response:
            response.raise_for_status()
            total = int(response.headers.get('content-length', 0))
            with archive.open('wb') as handle, tqdm(total=total, unit='B', unit_scale=True) as bar:
                for block in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if block:
                        handle.write(block)
                        bar.update(len(block))
    extract_marker = DATA_DIR / '.extracted'
    if not extract_marker.exists():
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(DATA_DIR)
        extract_marker.write_text('ok', encoding='utf-8')
    search_root = DATA_DIR

videos = sorted(search_root.rglob('*.mp4'))
pattern = re.compile(r'Cycle_(\d+)_task_(\d+)\.mp4$', re.I)
records = []
for path in videos:
    match = pattern.search(path.name)
    if not match:
        continue
    cycle_id, task_id = map(int, match.groups())
    records.append({
        'sample_id': f'hatrec_c{cycle_id:03d}_t{task_id}',
        'cycle_id': cycle_id,
        'task_id': task_id,
        'task_name': TASK_CATALOG[task_id],
        'source_path': str(path.resolve()),
        'source_sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
    })

assert len(records) == 78 * 7, f'Expected 546 clips, found {len(records)}'
assert sorted({r['cycle_id'] for r in records}) == list(range(78))
assert sorted({r['task_id'] for r in records}) == list(range(7))
assert len({r['source_sha256'] for r in records}) == len(records), 'Exact duplicate videos detected'

manifest_path = OUTPUT_DIR / 'hatrec_manifest.jsonl'
manifest_path.write_text(
    ''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in records),
    encoding='utf-8',
)
print({'clips': len(records), 'cycles': 78, 'classes': 7,
       'clips_per_class': {task: sum(r['task_id'] == task for r in records) for task in TASK_CATALOG},
       'exact_duplicates': 0, 'manifest': str(manifest_path)})


## 4. Inspect the pilot without exposing its label to the model

In [ ]:
import av
import pandas as pd
from IPython.display import Video, display

def video_metadata(path):
    with av.open(str(path)) as container:
        stream = container.streams.video[0]
        fps = float(stream.average_rate) if stream.average_rate else None
        duration = float(stream.duration * stream.time_base) if stream.duration else None
        if duration is None and container.duration:
            duration = float(container.duration / av.time_base)
        return {
            'duration_s': duration,
            'fps': fps,
            'width': stream.width,
            'height': stream.height,
            'frames_declared': stream.frames,
        }

pilot = next(r for r in records
             if r['cycle_id'] == PILOT_CYCLE and r['task_id'] == PILOT_TASK)
source_video = Path(pilot['source_path'])
blind_video = BLIND_DIR / 'sample_0001.mp4'
shutil.copy2(source_video, blind_video)
meta = video_metadata(blind_video)

print('Anonymous model input:', blind_video)
print(json.dumps(meta, indent=2))
display(Video(str(blind_video), embed=True, width=720))
print('Ground truth is stored outside the prompt and is revealed only after inference.')


## 5. Load only the Cosmos 3 Reasoner tower

In [ ]:
import torch
import transformers
from huggingface_hub import login
from transformers import AutoProcessor, Cosmos3OmniForConditionalGeneration

hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        pass
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    print('No HF_TOKEN found. Public access may work; gated access requires accepting the model license and adding HF_TOKEN.')

assert torch.cuda.is_available(), 'A CUDA GPU is required for this notebook.'
gpu_info = []
for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    gpu_info.append({'index': idx, 'name': props.name,
                     'vram_gb': round(props.total_memory / 2**30, 1),
                     'bf16': torch.cuda.is_bf16_supported()})
print('transformers:', transformers.__version__)
print('GPUs:', gpu_info)

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Cosmos3OmniForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    device_map='auto',
    low_cpu_mem_usage=True,
)
model.eval()
print('Reasoner loaded:', MODEL_ID, '| dtype:', dtype)


## 6. Evidence-grounded prompts, JSON parser and inference helpers

In [ ]:
SEGMENT_SCHEMA = {
    "segment_summary": "brief evidence-grounded summary",
    "observed": [
        {
            "relative_time": "start/middle/end or visible timestamp",
            "hand_action": "directly visible action or unknown",
            "objects_or_tools": ["visible object; generic if identity is uncertain"],
            "state_change": "visible before-to-after change or unknown",
            "evidence": "specific visual evidence"
        }
    ],
    "inferred": ["short interpretations, each supported by observed evidence"],
    "unknown": ["facts that cannot be established visually"],
    "occlusion_or_quality": ["blur, occlusion, poor view, missing frames"],
    "task_hypothesis": "free-form task description without dataset labels",
    "alternative_hypothesis": "plausible alternative or unknown",
    "execution_or_safety_observations": ["visible concern only; otherwise none"],
    "confidence": 0.0
}

FINAL_SCHEMA = {
    "video_overview": "what is directly visible across the complete video",
    "temporal_sequence": ["ordered action/state transitions"],
    "objects_and_tools": ["visible objects/tools"],
    "hand_object_interactions": ["grounded interactions"],
    "predicted_assembly_task": "free-form task description",
    "alternative_interpretation": "alternative or unknown",
    "evidence_supporting_prediction": ["segment-indexed evidence"],
    "execution_quality": "only what can be judged visually",
    "possible_errors": ["visible error candidates; not invented"],
    "safety_observations": ["visible safety observations"],
    "occlusions_and_limitations": ["limitations"],
    "final_synthesis": "concise final assessment",
    "overall_confidence": 0.0
}

SYSTEM_PROMPT = """You are an industrial assembly video analyst.
Use only evidence visible in the supplied video. Separate direct observations
from interpretations and unknowns. Never invent part identity, hidden machine
state, completion status, worker intent, safety compliance, or outcome. If the
evidence is insufficient, say unknown. Return valid JSON only, with no markdown
fences and no private chain-of-thought."""

def parse_json_response(text):
    cleaned = text.strip()
    cleaned = re.sub(r'^```(?:json)?\\s*', '', cleaned, flags=re.I)
    cleaned = re.sub(r'\\s*```$', '', cleaned)
    try:
        return json.loads(cleaned), True
    except json.JSONDecodeError:
        match = re.search(r'\\{.*\\}', cleaned, flags=re.S)
        if match:
            try:
                return json.loads(match.group(0)), True
            except json.JSONDecodeError:
                pass
    return {'raw_unparsed_response': text}, False

def generate_cosmos(messages, *, fps=None, max_new_tokens=700):
    kwargs = dict(tokenize=True, add_generation_prompt=True,
                  return_dict=True, return_tensors='pt')
    if fps is not None:
        kwargs['fps'] = fps
    inputs = processor.apply_chat_template(messages, **kwargs).to(model.device, dtype)
    started = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens,
        )
    elapsed = time.perf_counter() - started
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
    text = processor.batch_decode(trimmed, skip_special_tokens=True,
                                  clean_up_tokenization_spaces=False)[0]
    return text, elapsed

def analyze_segment(video_path, index, start_s, end_s):
    prompt = f"""Analyze temporal group {index + 1}, corresponding to
{start_s:.2f}-{end_s:.2f} seconds of an anonymized industrial assembly video.
Do not guess the HATRec class or use filename information.
Return exactly this JSON structure:
{json.dumps(SEGMENT_SCHEMA, ensure_ascii=False, indent=2)}"""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': [
            {'type': 'video', 'path': str(Path(video_path).resolve())},
            {'type': 'text', 'text': prompt},
        ]},
    ]
    raw, latency = generate_cosmos(messages, fps=VIDEO_FPS_FOR_MODEL,
                                   max_new_tokens=MAX_NEW_TOKENS_SEGMENT)
    parsed, valid = parse_json_response(raw)
    return {'group': index + 1, 'start_s': start_s, 'end_s': end_s,
            'json_valid': valid, 'latency_s': latency,
            'analysis': parsed, 'raw_response': raw}


## 7. Split into temporal groups and analyze each group

In [ ]:
def split_video(path, duration_s):
    target = CHUNK_DIR / 'sample_0001'
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True)
    n = max(1, min(MAX_SEGMENTS, math.ceil(duration_s / SEGMENT_SECONDS)))
    group_length = duration_s / n
    chunks = []
    for index in range(n):
        start = index * group_length
        end = duration_s if index == n - 1 else (index + 1) * group_length
        out = target / f'group_{index + 1:02d}.mp4'
        command = [
            'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
            '-ss', f'{start:.4f}', '-i', str(path), '-t', f'{end-start:.4f}',
            '-map_metadata', '-1', '-an', '-c:v', 'libx264', '-preset', 'ultrafast',
            '-crf', '18', '-pix_fmt', 'yuv420p', str(out)
        ]
        subprocess.run(command, check=True)
        chunks.append((out, start, end))
    return chunks

duration_s = meta['duration_s']
assert duration_s and duration_s > 0, 'Could not read video duration.'
chunks = split_video(blind_video, duration_s)
print('Temporal groups:', [(round(s, 2), round(e, 2)) for _, s, e in chunks])

group_results = []
for index, (chunk, start, end) in enumerate(chunks):
    result = analyze_segment(chunk, index, start, end)
    group_results.append(result)
    print(f"group {index+1}/{len(chunks)} | json={result['json_valid']} | "
          f"latency={result['latency_s']:.1f}s")

(OUTPUT_DIR / 'sample_0001_group_results.json').write_text(
    json.dumps(group_results, ensure_ascii=False, indent=2), encoding='utf-8')


## 8. Hierarchical full-video synthesis and report generation

In [ ]:
synthesis_input = [
    {k: value for k, value in group.items() if k != 'raw_response'}
    for group in group_results
]
synthesis_prompt = f"""Synthesize the temporal analyses below into one
evidence-grounded full industrial assembly report. Do not add facts not present
in the group analyses. Cite group numbers in evidence. The dataset label remains
hidden. Return exactly this JSON structure:
{json.dumps(FINAL_SCHEMA, ensure_ascii=False, indent=2)}

TEMPORAL GROUP ANALYSES:
{json.dumps(synthesis_input, ensure_ascii=False)}"""

synthesis_messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': [{'type': 'text', 'text': synthesis_prompt}]},
]
synthesis_raw, synthesis_latency = generate_cosmos(
    synthesis_messages, max_new_tokens=MAX_NEW_TOKENS_SYNTHESIS)
synthesis, synthesis_valid = parse_json_response(synthesis_raw)

def bullets(values, empty='- None established from visible evidence.'):
    if not values:
        return empty
    return '\n'.join(f'- {value}' for value in values)

def render_report(groups, final, meta):
    lines = [
        '# FULL INDUSTRIAL ASSEMBLY ACTION ANALYSIS REPORT', '',
        f"**Video:** `sample_0001.mp4` (anonymized HATRec clip)  ",
        f"**Duration:** {meta['duration_s']:.2f} s  ",
        f"**Model:** `{MODEL_ID}`  ",
        f"**Video sampling:** {VIDEO_FPS_FOR_MODEL} FPS  ",
        f"**Temporal groups:** {len(groups)}  ",
        '**Ground truth:** withheld until after inference', '',
        '---', ''
    ]
    for group in groups:
        analysis = group['analysis']
        lines += [
            f"## GROUP {group['group']} ({group['start_s']:.2f}–{group['end_s']:.2f}s)", '',
            f"**Segment summary:** {analysis.get('segment_summary', 'Unavailable')}", '',
            '**Direct observations:**',
        ]
        observed = analysis.get('observed', [])
        if observed:
            for item in observed:
                lines.append(
                    f"- **{item.get('relative_time', 'unknown time')}:** "
                    f"hand action={item.get('hand_action', 'unknown')}; "
                    f"objects/tools={item.get('objects_or_tools', [])}; "
                    f"state change={item.get('state_change', 'unknown')}; "
                    f"evidence={item.get('evidence', 'not supplied')}"
                )
        else:
            lines.append('- No parseable observations.')
        lines += ['', '**Inferred:**', bullets(analysis.get('inferred', [])), '',
                  '**Unknown / not visually established:**', bullets(analysis.get('unknown', [])), '',
                  '**Occlusion or image-quality limitations:**', bullets(analysis.get('occlusion_or_quality', [])), '',
                  f"**Task hypothesis:** {analysis.get('task_hypothesis', 'unknown')}",
                  f"**Alternative:** {analysis.get('alternative_hypothesis', 'unknown')}",
                  f"**Confidence:** {analysis.get('confidence', 'unknown')}", '', '---', '']

    lines += [
        '# FULL VIDEO SYNTHESIS', '',
        '## 1. Video overview', final.get('video_overview', 'Unavailable'), '',
        '## 2. Temporal sequence', bullets(final.get('temporal_sequence', [])), '',
        '## 3. Objects and tools', bullets(final.get('objects_and_tools', [])), '',
        '## 4. Hand–object interactions', bullets(final.get('hand_object_interactions', [])), '',
        '## 5. Predicted assembly task', final.get('predicted_assembly_task', 'unknown'), '',
        '## 6. Alternative interpretation', final.get('alternative_interpretation', 'unknown'), '',
        '## 7. Evidence supporting the prediction', bullets(final.get('evidence_supporting_prediction', [])), '',
        '## 8. Execution quality', final.get('execution_quality', 'unknown'), '',
        '## 9. Possible errors', bullets(final.get('possible_errors', [])), '',
        '## 10. Safety observations', bullets(final.get('safety_observations', [])), '',
        '## 11. Occlusions and limitations', bullets(final.get('occlusions_and_limitations', [])), '',
        '## 12. Final synthesis', final.get('final_synthesis', 'Unavailable'), '',
        f"**Overall confidence:** {final.get('overall_confidence', 'unknown')}", '',
        '> Claims in this report are model outputs and require human verification.',
    ]
    return '\n'.join(str(line) for line in lines)

report_text = render_report(group_results, synthesis, meta)
report_path = OUTPUT_DIR / 'sample_0001_cosmos3_report.md'
report_path.write_text(report_text, encoding='utf-8')
(OUTPUT_DIR / 'sample_0001_synthesis.json').write_text(
    json.dumps({'json_valid': synthesis_valid, 'latency_s': synthesis_latency,
                'analysis': synthesis, 'raw_response': synthesis_raw},
               ensure_ascii=False, indent=2), encoding='utf-8')

from IPython.display import Markdown
display(Markdown(report_text))
print('report:', report_path)


## 9. Closed-set task prediction — reveal candidate semantics, not ground truth

In [ ]:
CLASSIFICATION_SCHEMA = {
    "predicted_task_id": 0,
    "predicted_task_name": "one candidate name",
    "alternative_task_id": 1,
    "visible_evidence": ["short grounded evidence"],
    "uncertainty": "what remains unclear",
    "confidence": 0.0
}

def classify_video(path):
    candidates = '\n'.join(f'{key}: {value}' for key, value in TASK_CATALOG.items())
    prompt = f"""Classify this anonymized HATRec assembly clip using exactly
one candidate. Use only visible video evidence; filenames and the ground truth
are unavailable.

CANDIDATES:
{candidates}

Return exactly this JSON structure:
{json.dumps(CLASSIFICATION_SCHEMA, indent=2)}"""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': [
            {'type': 'video', 'path': str(Path(path).resolve())},
            {'type': 'text', 'text': prompt},
        ]},
    ]
    raw, latency = generate_cosmos(messages, fps=VIDEO_FPS_FOR_MODEL,
                                   max_new_tokens=350)
    parsed, valid = parse_json_response(raw)
    return {'json_valid': valid, 'latency_s': latency,
            'analysis': parsed, 'raw_response': raw}

classification = classify_video(blind_video)
predicted_id = classification['analysis'].get('predicted_task_id')
try:
    predicted_id = int(predicted_id)
except (TypeError, ValueError):
    predicted_id = None

posthoc = {
    'sample_id': pilot['sample_id'],
    'ground_truth_task_id': pilot['task_id'],
    'ground_truth_task_name': pilot['task_name'],
    'predicted_task_id': predicted_id,
    'predicted_task_name': classification['analysis'].get('predicted_task_name'),
    'correct': predicted_id == pilot['task_id'],
    'confidence': classification['analysis'].get('confidence'),
    'json_valid': classification['json_valid'],
    'latency_s': classification['latency_s'],
}
(OUTPUT_DIR / 'sample_0001_posthoc_evaluation.json').write_text(
    json.dumps(posthoc, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(posthoc, ensure_ascii=False, indent=2))


## 10. Causal controls — black video and reversed time

In [ ]:
def build_controls(path, duration, width, height):
    controls = {}
    black = BLIND_DIR / 'control_black.mp4'
    reverse = BLIND_DIR / 'control_reversed.mp4'
    subprocess.run([
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-f', 'lavfi', '-i', f'color=c=black:s={width}x{height}:r=24',
        '-t', f'{duration:.4f}', '-an', '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
        str(black)
    ], check=True)
    subprocess.run([
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y', '-i', str(path),
        '-vf', 'reverse', '-map_metadata', '-1', '-an', '-c:v', 'libx264',
        '-preset', 'ultrafast', '-crf', '18', '-pix_fmt', 'yuv420p', str(reverse)
    ], check=True)
    controls['black'] = black
    controls['reversed'] = reverse
    return controls

control_results = []
if RUN_CAUSAL_CONTROLS:
    controls = build_controls(blind_video, meta['duration_s'], meta['width'], meta['height'])
    for control_name, control_path in controls.items():
        result = classify_video(control_path)
        analysis = result['analysis']
        try:
            control_pred = int(analysis.get('predicted_task_id'))
        except (TypeError, ValueError):
            control_pred = None
        control_results.append({
            'control': control_name,
            'predicted_task_id': control_pred,
            'confidence': analysis.get('confidence'),
            'same_prediction_as_real': control_pred == predicted_id,
            'json_valid': result['json_valid'],
            'latency_s': result['latency_s'],
            'raw_response': result['raw_response'],
        })

    real_conf = classification['analysis'].get('confidence')
    try:
        real_conf = float(real_conf)
    except (TypeError, ValueError):
        real_conf = None
    for row in control_results:
        try:
            row['confidence_gap_vs_real'] = real_conf - float(row['confidence'])
        except (TypeError, ValueError):
            row['confidence_gap_vs_real'] = None

(OUTPUT_DIR / 'sample_0001_causal_controls.json').write_text(
    json.dumps(control_results, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.DataFrame(control_results).drop(columns=['raw_response'], errors='ignore'))


## 11. Optional full-dataset zero-shot benchmark

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

benchmark_rows = []
if RUN_FULL_BENCHMARK:
    evaluation_records = records[:FULL_BENCHMARK_LIMIT]
    for index, row in enumerate(evaluation_records):
        alias = BLIND_DIR / f'benchmark_{index:04d}.mp4'
        shutil.copy2(row['source_path'], alias)
        result = classify_video(alias)
        try:
            prediction = int(result['analysis'].get('predicted_task_id'))
        except (TypeError, ValueError):
            prediction = -1
        benchmark_rows.append({
            'sample_id': row['sample_id'], 'cycle_id': row['cycle_id'],
            'ground_truth': row['task_id'], 'prediction': prediction,
            'confidence': result['analysis'].get('confidence'),
            'json_valid': result['json_valid'], 'latency_s': result['latency_s'],
        })
        alias.unlink(missing_ok=True)
        if (index + 1) % 10 == 0:
            print(f'{index+1}/{len(evaluation_records)}')

    benchmark_df = pd.DataFrame(benchmark_rows)
    valid_df = benchmark_df[benchmark_df.prediction.isin(TASK_CATALOG)]
    metrics = {
        'samples': len(benchmark_df),
        'json_valid_rate': float(benchmark_df.json_valid.mean()),
        'classification_valid_rate': float(len(valid_df) / len(benchmark_df)),
        'top1_accuracy_all': float((benchmark_df.prediction == benchmark_df.ground_truth).mean()),
        'macro_f1_all': float(f1_score(benchmark_df.ground_truth,
                                      benchmark_df.prediction,
                                      labels=list(TASK_CATALOG), zero_division=0)),
        'mean_latency_s': float(benchmark_df.latency_s.mean()),
    }
    print(json.dumps(metrics, indent=2))
    benchmark_df.to_json(OUTPUT_DIR / 'hatrec_predictions.jsonl', orient='records',
                         lines=True, force_ascii=False)
    (OUTPUT_DIR / 'hatrec_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')

    matrix = confusion_matrix(benchmark_df.ground_truth, benchmark_df.prediction,
                              labels=list(TASK_CATALOG))
    plt.figure(figsize=(9, 7))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=[TASK_CATALOG[i] for i in TASK_CATALOG],
                yticklabels=[TASK_CATALOG[i] for i in TASK_CATALOG])
    plt.xlabel('Predicted'); plt.ylabel('Ground truth'); plt.title('Cosmos 3 on HATRec')
    plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=180); plt.show()
else:
    print('Skipped. Set RUN_FULL_BENCHMARK=True only after the one-video pilot is accepted.')


## 12. Human-review sheet and portable output

In [ ]:
review_row = {
    'sample_id': pilot['sample_id'],
    'task_prediction_correct_0_1': '',
    'objects_grounded_0_2': '',
    'hand_action_grounded_0_2': '',
    'temporal_sequence_grounded_0_2': '',
    'hallucinated_claim_count': '',
    'unsafe_overclaim_count': '',
    'reviewer_notes': '',
}
pd.DataFrame([review_row]).to_csv(OUTPUT_DIR / 'human_review.csv', index=False)

run_summary = {
    'model': MODEL_ID,
    'dataset': DATASET_SLUG,
    'dataset_license': 'CC BY-NC-ND 4.0',
    'sample': pilot['sample_id'],
    'video_metadata': meta,
    'group_count': len(group_results),
    'group_json_valid_rate': sum(r['json_valid'] for r in group_results) / len(group_results),
    'synthesis_json_valid': synthesis_valid,
    'posthoc': posthoc,
    'causal_controls': [{k: v for k, v in row.items() if k != 'raw_response'}
                        for row in control_results],
    'limitations': [
        'HATRec provides task labels, not ground-truth natural-language rationales.',
        'Explanation quality requires human evidence review.',
        'Generated causal-control videos are private evaluation artifacts and must not be redistributed.',
    ],
}
(OUTPUT_DIR / 'run_summary.json').write_text(
    json.dumps(run_summary, ensure_ascii=False, indent=2), encoding='utf-8')

archive_path = shutil.make_archive(str(WORK / 'hatrec_cosmos3_pilot_outputs'), 'zip', OUTPUT_DIR)
print('PORTABLE OUTPUT:', archive_path)
print('Files:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print('-', path.name, round(path.stat().st_size / 1024, 1), 'KiB')


## Acceptance rule

Do not call the pilot successful merely because the report is fluent.

The pilot is credible only if:

1. the predicted task matches the hidden ground truth;
2. visible objects and hand actions are correctly grounded;
3. evidence cites the correct temporal groups;
4. unsupported machine state/outcome claims are absent;
5. confidence drops materially on the black-video control;
6. temporal reasoning changes appropriately on reversed video;
7. a human reviewer can verify the final synthesis against the source clip.

The paper's HATREC hybrid YOLO+LSTM system reports approximately **85.23%**
frame-level task-recognition accuracy. Treat that as a domain-specific reference,
not a directly equivalent zero-shot comparison.
